# Can we predict next-month churn?
Target: `churn_next_month` (monthly account snapshots, 3,000 accounts). Plan:
1. Inspect the data; look for leakage and structure that affects validation.
2. Build features using only information available at snapshot time.
3. Validate **out-of-time** (train on earlier months, test on later ones).
4. Compare against simple baselines and report honestly.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import GroupKFold
from sklearn.inspection import permutation_importance
pd.set_option('display.width', 140)
D = '/private/tmp/waldean-agent-baseline/runs-v3/subscriptions-claude-default-13/'
print(pd.read_csv(D + 'columns.csv').to_string())
df = pd.read_csv(D + 'data.csv').sort_values(['account_ref', 'snapshot_month']).reset_index(drop=True)
print(df.shape, 'base rate', round(df.churn_next_month.mean(), 4))
df.head()

## 1. Data checks

In [ ]:
print(df.isna().sum().sum(), 'missing;', df.duplicated(['account_ref', 'snapshot_month']).sum(), 'duplicate account-months')
print(df.groupby('account_ref').size().describe().round(1))
last = df.groupby('account_ref').snapshot_month.transform('max') == df.snapshot_month
print(pd.crosstab(df.churn_next_month, last, rownames=['churn'], colnames=['is last snapshot']))
df.groupby('snapshot_month').churn_next_month.mean().plot(kind='bar', figsize=(10, 3), title='Churn rate by snapshot month')
plt.show()

Observations:
* **Churn rate swings by calendar month**: ~3% in spring 2024, >50% in Dec 2024 - Jan 2025, ~10% afterwards. This period effect (cause unknown) makes random splits over-optimistic.
* Accounts appear many times, so rows are not independent; splits must respect time (and accounts).

### Leakage check: `lifetime_months`
The dictionary says it is the *total months the account was active, from a CRM export of 2026-01-05*, i.e. measured **after** the snapshot. It would not be known at prediction time. Let's see how it relates to the label.

In [ ]:
df['months_left'] = df.lifetime_months - df.tenure_months
print(pd.crosstab(df.months_left.clip(upper=5), df.churn_next_month))

`lifetime_months - tenure_months` is 0 for every churner and positive for non-churners, so it essentially encodes the label. **It is excluded from the model**; a comparison below shows how much it would inflate results.

## 2. Features (available at snapshot time only)

In [ ]:
df['month'] = pd.PeriodIndex(df.snapshot_month, freq='M')
df['fee_per_seat'] = df.monthly_fee / df.seats
g = df.groupby('account_ref')
# history features use earlier snapshots of the same account only
df['usage_prev'] = g.usage_hours.shift(1)
df['usage_chg'] = df.usage_hours - df.usage_prev
df['usage_ratio_prev'] = (df.usage_hours / df.usage_prev).replace([np.inf, -np.inf], np.nan)
df['tickets_prev'] = g.support_tickets.shift(1)
df['usage_roll3'] = g.usage_hours.transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
df['fee_chg'] = df.monthly_fee - g.monthly_fee.shift(1)
prev_plan = g.plan.shift(1)
df['plan_chg'] = (prev_plan.notna() & (prev_plan != df.plan)).astype(int)
plan_d = pd.get_dummies(df.plan, prefix='plan', dtype=int)
df = pd.concat([df, plan_d], axis=1)
base = ['seats', 'monthly_fee', 'fee_per_seat', 'tenure_months', 'usage_hours', 'support_tickets'] + list(plan_d.columns)
hist = ['usage_chg', 'usage_ratio_prev', 'tickets_prev', 'usage_roll3', 'fee_chg', 'plan_chg']
df.groupby('churn_next_month')[['seats', 'monthly_fee', 'tenure_months', 'usage_hours', 'support_tickets']].mean().T

## 3. Validation design
Out-of-time: train on snapshots up to 2025-02, skip 2025-03 (its label month, April, overlaps the test window), **test on snapshots 2025-04 to 2025-11**. Train still includes the 2024-25 spike; that is realistic, since that is the history we would have.

In [ ]:
cut = pd.Period('2025-04', 'M')
tr = df[df.month < cut - 1]; te = df[df.month >= cut]
print(len(tr), len(te), 'train churn', round(tr.churn_next_month.mean(), 3), 'test churn', round(te.churn_next_month.mean(), 3))
ytr, yte = tr.churn_next_month, te.churn_next_month

def lift10(score, y):
    k = int(len(y) * .1); return y.values[np.argsort(-np.asarray(score))[:k]].mean() / y.mean()
def fit_eval(name, model, feats):
    model.fit(tr[feats].fillna(0), ytr); p = model.predict_proba(te[feats].fillna(0))[:, 1]
    return dict(model=name, AUC=roc_auc_score(yte, p), AP=average_precision_score(yte, p), lift_top10pct=lift10(p, yte))
lr = lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
gb = lambda: HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0)
rows = [fit_eval('LogReg: current snapshot', lr(), base), fit_eval('LogReg: + history', lr(), base + hist),
        fit_eval('GBM: current snapshot', gb(), base), fit_eval('GBM: + history', gb(), base + hist)]
for n, s in [('Baseline: low tenure', -te.tenure_months), ('Baseline: low usage', -te.usage_hours)]:
    rows.append(dict(model=n, AUC=roc_auc_score(yte, s), AP=average_precision_score(yte, s), lift_top10pct=lift10(s, yte)))
rows.append(dict(model='Random (base rate)', AUC=0.5, AP=yte.mean(), lift_top10pct=1.0))
pd.DataFrame(rows).round(3)

### Same model with the leaky `lifetime_months` (do NOT use) - for comparison only

In [ ]:
f = base + hist + ['lifetime_months']
m = gb().fit(tr[f].fillna(0), ytr); p = m.predict_proba(te[f].fillna(0))[:, 1]
print('GBM incl. lifetime_months  AUC=%.3f  AP=%.3f' % (roc_auc_score(yte, p), average_precision_score(yte, p)))

## 4. Robustness, drivers, calibration

In [ ]:
f = base + hist
X, yy = df[f].fillna(0), df.churn_next_month
aucs = [roc_auc_score(yy.iloc[b], gb().fit(X.iloc[a], yy.iloc[a]).predict_proba(X.iloc[b])[:, 1])
        for a, b in GroupKFold(5).split(X, yy, df.account_ref)]
print('Account-grouped 5-fold AUC (mixes periods): %.3f +/- %.3f' % (np.mean(aucs), np.std(aucs)))

mm = gb().fit(tr[f].fillna(0), ytr)
te = te.assign(p=mm.predict_proba(te[f].fillna(0))[:, 1])
print(te.groupby('snapshot_month').apply(lambda d: pd.Series(dict(
    n=len(d), churn=d.churn_next_month.mean(), AUC=roc_auc_score(d.churn_next_month, d.p))), include_groups=False).round(3))

pi = permutation_importance(mm, te[f].fillna(0), yte, scoring='roc_auc', n_repeats=5, random_state=0)
pd.Series(pi.importances_mean, f).sort_values().plot(kind='barh', figsize=(6, 4), title='Permutation importance (test AUC drop)')
plt.show()
te['decile'] = pd.qcut(te.p, 10, duplicates='drop')
te.groupby('decile', observed=True).agg(pred=('p', 'mean'), actual=('churn_next_month', 'mean'), n=('p', 'size')).round(3)

## 5. Conclusions
(Numbers from the run on 2026-10-13; re-run to refresh.)
* **Modest, real signal.** Out-of-time (test 2025-04..11, 9.6% churn) AUC is ~0.66-0.68 and the top-10% riskiest snapshots churn ~2.3-2.7x the base rate. A plain logistic regression is as good as gradient boosting; history features did not help. Account-grouped CV gives ~0.71 (easier, since it mixes periods).
* **Drivers**: lower usage and more support tickets precede churn; tenure alone is not predictive.
* **Not good enough for precise individual-level prediction**: most churners are not in the top decile. Fine for prioritising outreach, not for forecasting.
* **`lifetime_months` is target leakage** (AUC 0.99 with it); it was excluded. Do not use it.
* **Calibration**: the model over-predicts on the test period (mean prediction ~0.23 vs 9.6% actual) because training includes the Nov 2024-Jan 2025 spike. Ranking holds; absolute probabilities need recalibration on recent data.
* Churn rate has a strong period effect whose cause is not in the data; that is the biggest unknown for production use.